In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    current_timestamp,
    from_json,
    lit,
    month,
    dayofmonth,
    hour,
    to_timestamp,
    when,
    year,
)
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)


KAFKA_BOOTSTRAP_SERVERS = "YOUR_KAFKA_PRIVATE_IP:9092"
KAFKA_TOPIC = "stock-trades-raw"

BRONZE_PATH = (
    "s3a://<Name of S3 Bucket>/"
    "stock-market/bronze/trades/"
)

CHECKPOINT_PATH = (
    "s3a://<Name of S3 Bucket>/"
    "stock-market/checkpoints/bronze_trades/"
)

ALLOWED_SYMBOLS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "TSLA",
    "AMZN",
]


TRADE_SCHEMA = StructType(
    [
        StructField("schema_version", StringType(), True),
        StructField("event_id", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("source", StringType(), True),
        StructField("symbol", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("volume", DoubleType(), True),
        StructField(
            "trade_conditions",
            ArrayType(StringType()),
            True,
        ),
        StructField("event_timestamp_ms", LongType(), True),
        StructField("event_timestamp_utc", StringType(), True),
        StructField("ingestion_timestamp_utc", StringType(), True),
    ]
)


def main():

    spark = (
        SparkSession.builder
        .appName("StockMarketBronzeWriter")
        .config("spark.sql.session.timeZone", "UTC")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    # -----------------------------------------------------
    # Read Kafka stream
    # -----------------------------------------------------

    raw_stream = (
        spark.readStream
        .format("kafka")
        .option(
            "kafka.bootstrap.servers",
            KAFKA_BOOTSTRAP_SERVERS,
        )
        .option("subscribe", KAFKA_TOPIC)

        # We want to process existing Kafka records for
        # the first Bronze load.
        .option("startingOffsets", "earliest")

        # Prevent a large Kafka backlog from overwhelming
        # our small EC2 Spark instance.
        .option("maxOffsetsPerTrigger", 2000)

        .load()
    )

    # -----------------------------------------------------
    # Convert Kafka binary data to strings
    # -----------------------------------------------------

    string_stream = raw_stream.selectExpr(
        "CAST(key AS STRING) AS message_key",
        "CAST(value AS STRING) AS raw_message",
        "topic",
        "partition",
        "offset",
        "timestamp AS kafka_timestamp",
    )

    # -----------------------------------------------------
    # Parse Finnhub JSON
    # -----------------------------------------------------

    parsed_stream = (
        string_stream
        .withColumn(
            "trade",
            from_json(
                col("raw_message"),
                TRADE_SCHEMA,
            ),
        )
        .select(
            col("trade.*"),
            col("raw_message"),
            col("message_key"),
            col("topic"),
            col("partition"),
            col("offset"),
            col("kafka_timestamp"),
        )
    )

    # -----------------------------------------------------
    # Convert timestamps
    # -----------------------------------------------------

    timestamped_stream = (
        parsed_stream
        .withColumn(
            "event_timestamp",
            to_timestamp(
                col("event_timestamp_utc")
            ),
        )
        .withColumn(
            "ingestion_timestamp",
            to_timestamp(
                col("ingestion_timestamp_utc")
            ),
        )
        .withColumn(
            "processing_timestamp",
            current_timestamp(),
        )
    )

    # -----------------------------------------------------
    # Validate records
    # -----------------------------------------------------

    validated_stream = (
        timestamped_stream
        .withColumn(
            "validation_error",
            when(
                col("event_id").isNull(),
                "missing_event_id",
            )
            .when(
                col("symbol").isNull(),
                "missing_symbol",
            )
            .when(
                ~col("symbol").isin(ALLOWED_SYMBOLS),
                "unsupported_symbol",
            )
            .when(
                col("message_key") != col("symbol"),
                "key_symbol_mismatch",
            )
            .when(
                col("price").isNull()
                | (col("price") <= 0),
                "invalid_price",
            )
            .when(
                col("volume").isNull()
                | (col("volume") <= 0),
                "invalid_volume",
            )
            .when(
                col("event_timestamp").isNull(),
                "invalid_event_timestamp",
            )
            .when(
                col("ingestion_timestamp").isNull(),
                "invalid_ingestion_timestamp",
            )
            .otherwise(
                lit(None).cast("string")
            ),
        )
    )

    # -----------------------------------------------------
    # Keep valid records only
    # -----------------------------------------------------

    valid_stream = (
        validated_stream
        .filter(
            col("validation_error").isNull()
        )
    )

    # -----------------------------------------------------
    # Create S3 partition columns
    #
    # Bronze is partitioned by ingestion time because
    # this represents when data entered our pipeline.
    # -----------------------------------------------------

    bronze_stream = (
        valid_stream
        .withColumn(
            "year",
            year(col("ingestion_timestamp")),
        )
        .withColumn(
            "month",
            month(col("ingestion_timestamp")),
        )
        .withColumn(
            "day",
            dayofmonth(col("ingestion_timestamp")),
        )
        .withColumn(
            "hour",
            hour(col("ingestion_timestamp")),
        )
    )

    # -----------------------------------------------------
    # Write continuously to Amazon S3
    # -----------------------------------------------------

    query = (
        bronze_stream.writeStream
        .format("parquet")
        .outputMode("append")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH,
        )
        .partitionBy(
            "year",
            "month",
            "day",
            "hour",
        )
        .trigger(
            processingTime="10 seconds"
        )
        .start(BRONZE_PATH)
    )

    print("==========================================")
    print("Bronze streaming writer started")
    print(f"Kafka topic: {KAFKA_TOPIC}")
    print(f"Bronze path: {BRONZE_PATH}")
    print(f"Checkpoint: {CHECKPOINT_PATH}")
    print("Press Ctrl+C to stop")
    print("==========================================")

    query.awaitTermination()


if __name__ == "__main__":
    main()